# Amazon Baby Reviews — Data Analysis

This notebook analyzes `amazon_baby.csv` using **pandas**, **NumPy**, and **matplotlib.pyplot**.

The dataset contains three columns:
- `name` — product name
- `review` — text of a customer review
- `rating` — rating from 1 to 5

Place `amazon_baby.csv` in the same folder as this notebook before running the cells.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: make plots a little larger
plt.rcParams["figure.figsize"] = (8, 5)

## Exercise 1

a) Load the reviews from `amazon_baby.csv` and print the first records.  
b) Print the variable list. Select categorical and numerical variables.  
c) Compute basic statistics of numerical variables. Are all of them meaningful?

In [ ]:
# a)
df = pd.read_csv("amazon_baby.csv")
df.head()

In [ ]:
# b)
print("List of variables:")
print(df.columns.tolist())

categorical = df.select_dtypes(include=["object", "category", "str"]).columns.tolist()
numerical = df.select_dtypes(include=["number"]).columns.tolist()

print("\nCategorical variables:")
print(categorical)

print("\nNumerical variables:")
print(numerical)

In [ ]:
# c)
print("Basic statistics of numerical variables:")
df[numerical].describe()

**Question:** The `rating` statistics are useful, but the dataset does not contain other numeric measurements by default. In later exercises, we will create a numeric variable describing the length of each review.

## Exercise 2

Using scatter plots:

a) Show the relation between review length and rating.  
b) Check how product ratings differ. Find products with the highest average rating, considering only products with enough reviews.  
c) Compare the number of reviews and average rating for products.

In [ ]:
# a) Create a review-length variable, measured in characters.
df["review_length"] = df["review"].fillna("").str.len()

plt.scatter(df["review_length"], df["rating"], alpha=0.15)
plt.xlabel("Review length (characters)")
plt.ylabel("Rating")
plt.title("Review length versus rating")
plt.show()

In [ ]:
# b) Keep products with at least 100 reviews to avoid ranking products
# based on only one or two ratings.
product_stats = df.groupby("name", dropna=True).agg(
    number_of_reviews=("rating", "count"),
    average_rating=("rating", "mean")
)

popular_products = product_stats.loc[product_stats["number_of_reviews"] >= 100]
best_products = popular_products.sort_values("average_rating", ascending=False).head(10)

print("Products with the highest average rating (at least 100 reviews):")
best_products

In [ ]:
# c) Each point represents one product.
plt.scatter(
    product_stats["number_of_reviews"],
    product_stats["average_rating"],
    alpha=0.35
)
plt.xlabel("Number of reviews")
plt.ylabel("Average rating")
plt.title("Product popularity versus average rating")
plt.show()

## Exercise 3

a) Create a boxplot of rating versus review-length group. Compare it with Exercise 2a.  
b) Create a mosaic-like plot showing the relationship between rating category and review-length category.

In [ ]:
# a) Divide reviews into three length groups.
df["review_length_group"] = pd.cut(
    df["review_length"],
    bins=[-1, 50, 200, np.inf],
    labels=["Short (0–50)", "Medium (51–200)", "Long (201+)"]
)

df.boxplot(column="rating", by="review_length_group")
plt.xlabel("Review length group")
plt.ylabel("Rating")
plt.title("Rating by review length group")
plt.suptitle("")
plt.show()

In [ ]:
# b) Make categories from the numeric rating and review length.
df["rating_category"] = pd.cut(
    df["rating"],
    bins=[0, 2, 3, 5],
    labels=["Low (1–2)", "Middle (3)", "High (4–5)"]
)

counts = pd.crosstab(df["review_length_group"], df["rating_category"])
counts.plot(kind="bar", stacked=True)
plt.xlabel("Review length group")
plt.ylabel("Number of reviews")
plt.title("Rating category by review length group")
plt.legend(title="Rating category")
plt.tight_layout()
plt.show()

**Note:** This is a stacked bar chart of the same category counts that a mosaic plot would help compare. It uses only pandas and matplotlib, so no extra plotting package is required.

## Exercise 4

“I am interested in products that have at least 100 reviews.”

a) Compute basic statistics for reviews of these products.  
b) Highlight these products on a plot of number of reviews versus average rating.  
c) Find products with the best average rating among products with at least 100 reviews. Also show the number of reviews for each.

In [ ]:
# a) Select products with at least 100 reviews.
product_stats = df.groupby("name", dropna=True).agg(
    number_of_reviews=("rating", "count"),
    average_rating=("rating", "mean"),
    median_rating=("rating", "median")
)

selected_products = product_stats.loc[product_stats["number_of_reviews"] >= 100]

print("Number of products with at least 100 reviews:", len(selected_products))
selected_products.describe()

In [ ]:
# b) Show all products and highlight products with at least 100 reviews.
plt.scatter(
    product_stats["number_of_reviews"],
    product_stats["average_rating"],
    alpha=0.2,
    label="All products"
)
plt.scatter(
    selected_products["number_of_reviews"],
    selected_products["average_rating"],
    alpha=0.7,
    label="At least 100 reviews"
)
plt.xlabel("Number of reviews")
plt.ylabel("Average rating")
plt.title("Products with at least 100 reviews")
plt.legend()
plt.show()

In [ ]:
# c) Sort selected products by average rating.
# Use number of reviews as a secondary sort key.
best_products = selected_products.sort_values(
    ["average_rating", "number_of_reviews"],
    ascending=[False, False]
).head(10)

print(best_products[["number_of_reviews", "average_rating", "median_rating"]])

## Additional observations

- Missing product names and reviews can affect grouping and review-length calculations.
- An average rating based on many reviews is generally more informative than an average based on very few reviews.
- Review length does not necessarily measure review quality or customer satisfaction.
- These results describe associations in the data; they do not prove that one variable causes another.